<a href="https://colab.research.google.com/github/biglalo104/Projects/blob/main/MapReduce%20job%20that%20calculates%20statistics%20like%20population%20totals%20and%20averages%20per%20county.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Create a dedicated directory in HDFS for the census data
!hdfs dfs -mkdir -p /user/hadoop/kenya_census_2019

# Upload the local CSV file to HDFS
!hdfs dfs -put 2019-population_census-report-per-county.csv /user/hadoop/kenya_census_2019/

# Verify the upload and check file permissions
!hdfs dfs -ls /user/hadoop/kenya_census_2019/

/bin/bash: line 1: hdfs: command not found
/bin/bash: line 1: hdfs: command not found
/bin/bash: line 1: hdfs: command not found


In [3]:
!pip install pyspark

In [4]:
from pyspark.sql import SparkSession

# Initialize a Spark session in local mode
spark = SparkSession.builder \
    .appName("KenyaCensusAnalysis") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session successfully initialized in local mode:", spark)

Spark Session successfully initialized in local mode: <pyspark.sql.session.SparkSession object at 0x783f802f5d30>


In [5]:
# Load the local census CSV file into a Spark DataFrame
file_path = "/content/2019-population_census-report-per-county.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)

# Show the schema and the first few rows
df.printSchema()
display(df.limit(5).toPandas())

root
 |-- County: string (nullable = true)
 |-- Total_Population19: string (nullable = true)
 |-- Male populatio 2019: string (nullable = true)
 |-- Female population 2019: string (nullable = true)
 |-- Households: string (nullable = true)
 |-- Av_HH_Size: double (nullable = true)
 |-- LandArea: string (nullable = true)
 |-- Population Density: string (nullable = true)
 |-- Population in 2009: string (nullable = true)
 |-- Pop_change: string (nullable = true)
 |-- Intersex population 2019: double (nullable = true)



,County,Total_Population19,Male populatio 2019,Female population 2019,Households,Av_HH_Size,LandArea,Population Density,Population in 2009,Pop_change,Intersex population 2019
0,Baringo,"666,763","336,322","330,428","142,518",5.0,"10,976",61,"555,561","111,202",13.0
1,Bomet,"875,689","434,287","441,379","187,641",5.0,"2,531",346,"724,186","151,503",23.0
2,Bungoma,"1,670,570","812,146","858,389","358,796",5.0,"3,024",552,"1,630,934","39,636",35.0
3,Busia,"893,681","426,252","467,401","198,152",5.0,"1,696",527,"488,075","405,606",28.0
4,Elgeyo-Marakwet,"454,480","227,317","227,151","99,861",5.0,"3,032",150,"369,998","84,482",12.0


In [6]:
#!/usr/bin/env python
import sys

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    # Split by comma, but be mindful that some fields might be quoted
    # For simplicity, we split by comma and clean individual fields
    parts = line.split(',')

    # Ensure we have enough columns and skip the header row
    if len(parts) >= 5 and parts[0].strip().replace('"', '') != 'County':
        county = parts[0].strip().replace('"', '')

        try:
            # Clean numeric fields: remove quotes, commas, and spaces, then convert to integer
            total_pop = int(parts[1].strip().replace('"', '').replace(',', '').replace(' ', ''))
            households = int(parts[4].strip().replace('"', '').replace(',', '').replace(' ', ''))

            # Emit: County \t Total_Population \t Households
            print(f"{county}\t{total_pop}\t{households}")
        except ValueError:
            # Skip rows with malformed numeric data
            continue

In [7]:
#!/usr/bin/env python
import sys

current_county = None
total_pop = 0
total_households = 0

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    parts = line.split('\t')
    if len(parts) == 3:
        county = parts[0]
        pop = int(parts[1])
        hh = int(parts[2])

        if current_county == county:
            # Aggregate in case of duplicate county entries
            total_pop += pop
            total_households += hh
        else:
            # Print the result for the previous county before moving to the new one
            if current_county:
                avg_hh_size = total_pop / total_households if total_households > 0 else 0.0
                print(f"{current_county}\t{total_pop}\t{total_households}\t{avg_hh_size:.2f}")

            # Reset for the new county
            current_county = county
            total_pop = pop
            total_households = hh

# Print the final county's aggregated result
if current_county:
    avg_hh_size = total_pop / total_households if total_households > 0 else 0.0
    print(f"{current_county}\t{total_pop}\t{total_households}\t{avg_hh_size:.2f}")

In [9]:
from pyspark.sql import functions as F

# Perform the same aggregation logic using PySpark
# 1. Clean the numeric columns (remove commas and cast to Integer)
cleaned_df = df.withColumn(
    "Total_Population19",
    F.regexp_replace(F.col("Total_Population19"), "[\\s,]", "").cast("int")
).withColumn(
    "Households",
    F.regexp_replace(F.col("Households"), "[\\s,]", "").cast("int")
)

# 2. Aggregate: Group by County, sum populations/households, and calculate Average Household Size
result_df = cleaned_df.groupBy("County").agg(
    F.sum("Total_Population19").alias("Total_Population"),
    F.sum("Households").alias("Total_Households")
).withColumn(
    "Avg_Household_Size",
    F.round(F.col("Total_Population") / F.col("Total_Households"), 2)
)

# Display the results
result_df.show(5)

+-----------+----------------+----------------+------------------+
|     County|Total_Population|Total_Households|Avg_Household_Size|
+-----------+----------------+----------------+------------------+
|Uasin Gishu|         1163186|          304943|              3.81|
|     Migori|         1116436|          240168|              4.65|
|     Nakuru|         2162202|          616046|              3.51|
|   Laikipia|          518580|          149271|              3.47|
|       Embu|          608599|          182743|              3.33|
+-----------+----------------+----------------+------------------+
only showing top 5 rows


In [11]:
# Show the full results from our PySpark computation
result_df.orderBy("County").show(n=100, truncate=False)

+---------------+----------------+----------------+------------------+
|County         |Total_Population|Total_Households|Avg_Household_Size|
+---------------+----------------+----------------+------------------+
|Baringo        |666763          |142518          |4.68              |
|Bomet          |875689          |187641          |4.67              |
|Bungoma        |1670570         |358796          |4.66              |
|Busia          |893681          |198152          |4.51              |
|Elgeyo-Marakwet|454480          |99861           |4.55              |
|Embu           |608599          |182743          |3.33              |
|Garissa        |841353          |141394          |5.95              |
|Homa Bay       |1131950         |262036          |4.32              |
|Isiolo         |268002          |58072           |4.61              |
|Kajiado        |1117840         |316179          |3.54              |
|Kakamega       |1867579         |433207          |4.31              |
|Keric